In [1]:
import optuna
import pandas as pd
from sklearn.model_selection import cross_val_score, KFold, StratifiedShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from datetime import datetime
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import roc_auc_score
import numpy as np
import json

Model 1:

In [2]:
def objective_1(trial, features, target):
    params = {
        'max_depth': trial.suggest_int('max_depth', 2, 32, log=True),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 200),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 300),
        'bootstrap': True,
        'random_state': 20240627,  
    }

    model = RandomForestClassifier(**params)

    number_folds = 5
    n_splits = 5
    seed = 20240627
    cv = StratifiedShuffleSplit(n_splits=n_splits,
                                test_size=1/n_splits,
                                random_state=seed)
    auc_scores = cross_val_score(model, features, target, cv=cv, scoring='roc_auc')
    mean_auc = auc_scores.mean()
    std_err = np.std(auc_scores) / np.sqrt(number_folds)

    trial.set_user_attr("std_err", float(std_err)) 
    return mean_auc

In [3]:
if __name__ == "__main__":
    start_time = datetime.now()
    
    # choose model data
    model_id = "model_1"
    model_category = "rf"
    n_trials = 50
    
    # load training set
    X_data = pd.read_csv(f"data/X_train_{model_id}.csv", keep_default_na=False)
    y_data = pd.read_csv(f"data/y_train_{model_id}.csv", keep_default_na=False).values.ravel()

    # load study
    study = optuna.create_study(direction="maximize", 
                                sampler=optuna.samplers.TPESampler(seed=2020))
    
    # optimize calculate
    study.optimize(lambda trial: objective_1(trial, X_data, y_data), n_trials=n_trials, n_jobs=-1)
    
    # store into json file
    all_trials_path = f"results/{model_id}_{model_category}_trials.json"
    with open(all_trials_path, "w", encoding="utf-8") as outfile:
        for trial in study.trials:
            serialized_trial = {
                "number": trial.number,
                "value": trial.value,
                "params": trial.params,
                "user_attrs": trial.user_attrs,
            }
            json.dump(serialized_trial, outfile)
            outfile.write("\n")
    # find the best and store
    best_trial_path = f"results/{model_id}_{model_category}_trial_best.json"
    best_trial = study.best_trial
    serialized_best = {
        "number": best_trial.number,
        "value": best_trial.value,
        "params": best_trial.params,
        "user_attrs": best_trial.user_attrs,
    }
    with open(best_trial_path, "w", encoding="utf-8") as outfile:
        json.dump(serialized_best, outfile)
    print(f"Time: {datetime.now() - start_time}")

[I 2026-05-11 10:57:35,880] A new study created in memory with name: no-name-11bf53aa-c25a-4f79-80da-7ebdaf1dccff
[I 2026-05-11 10:58:07,957] Trial 15 finished with value: 0.8684291030459453 and parameters: {'max_depth': 3, 'min_samples_leaf': 20, 'min_samples_split': 199}. Best is trial 15 with value: 0.8684291030459453.
[I 2026-05-11 10:58:09,290] Trial 14 finished with value: 0.8740584149500306 and parameters: {'max_depth': 4, 'min_samples_leaf': 110, 'min_samples_split': 113}. Best is trial 14 with value: 0.8740584149500306.
[I 2026-05-11 10:58:09,627] Trial 1 finished with value: 0.874767733075007 and parameters: {'max_depth': 4, 'min_samples_leaf': 33, 'min_samples_split': 143}. Best is trial 1 with value: 0.874767733075007.
[I 2026-05-11 10:58:10,656] Trial 17 finished with value: 0.8775818381173905 and parameters: {'max_depth': 6, 'min_samples_leaf': 195, 'min_samples_split': 85}. Best is trial 17 with value: 0.8775818381173905.
[I 2026-05-11 10:58:10,976] Trial 9 finished with

Time: 0:01:43.477860


Model 2:

In [4]:
def objective_2(trial, features, target):
    params = {
        'max_depth': trial.suggest_int('max_depth', 2, 32, log=True),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 200),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'bootstrap': True,
        'random_state': 20240627,  
    }

    model = RandomForestClassifier(**params)

    number_folds = 5
    n_splits = 5
    seed = 20240627
    cv = StratifiedShuffleSplit(n_splits=n_splits,
                                test_size=1/n_splits,
                                random_state=seed)
    auc_scores = cross_val_score(model, features, target, cv=cv, scoring='roc_auc')
    mean_auc = auc_scores.mean()
    std_err = np.std(auc_scores) / np.sqrt(number_folds)

    trial.set_user_attr("std_err", float(std_err)) 
    return mean_auc

In [5]:
if __name__ == "__main__":
    start_time = datetime.now()
    
    # choose model data
    model_id = "model_2"
    model_category = "rf"
    n_trials = 50
    
    # load training set
    X_data = pd.read_csv(f"data/X_train_{model_id}.csv", keep_default_na=False)
    y_data = pd.read_csv(f"data/y_train_{model_id}.csv", keep_default_na=False).values.ravel()

    # load study
    study = optuna.create_study(direction="maximize", 
                                sampler=optuna.samplers.TPESampler(seed=2020))
    
    # optimize calculate
    study.optimize(lambda trial: objective_2(trial, X_data, y_data), n_trials=n_trials, n_jobs=-1)
    
    # store into json file
    all_trials_path = f"results/{model_id}_{model_category}_trials.json"
    with open(all_trials_path, "w", encoding="utf-8") as outfile:
        for trial in study.trials:
            serialized_trial = {
                "number": trial.number,
                "value": trial.value,
                "params": trial.params,
                "user_attrs": trial.user_attrs,
            }
            json.dump(serialized_trial, outfile)
            outfile.write("\n")
    # find the best and store
    best_trial_path = f"results/{model_id}_{model_category}_trial_best.json"
    best_trial = study.best_trial
    serialized_best = {
        "number": best_trial.number,
        "value": best_trial.value,
        "params": best_trial.params,
        "user_attrs": best_trial.user_attrs,
    }
    with open(best_trial_path, "w", encoding="utf-8") as outfile:
        json.dump(serialized_best, outfile)
    print(f"Time: {datetime.now() - start_time}")

[I 2026-05-11 10:59:19,404] A new study created in memory with name: no-name-dd36adb5-673f-43c8-a5ec-c19da2d16b3e
[I 2026-05-11 10:59:40,877] Trial 12 finished with value: 0.7872088390135564 and parameters: {'max_depth': 2, 'min_samples_leaf': 150, 'max_features': 'log2'}. Best is trial 12 with value: 0.7872088390135564.
[I 2026-05-11 10:59:41,204] Trial 15 finished with value: 0.7850411351847566 and parameters: {'max_depth': 2, 'min_samples_leaf': 105, 'max_features': 'sqrt'}. Best is trial 12 with value: 0.7872088390135564.
[I 2026-05-11 10:59:41,432] Trial 10 finished with value: 0.792740390509197 and parameters: {'max_depth': 3, 'min_samples_leaf': 75, 'max_features': 'log2'}. Best is trial 10 with value: 0.792740390509197.
[I 2026-05-11 10:59:41,983] Trial 14 finished with value: 0.7969409690421295 and parameters: {'max_depth': 4, 'min_samples_leaf': 2, 'max_features': 'log2'}. Best is trial 14 with value: 0.7969409690421295.
[I 2026-05-11 10:59:42,158] Trial 19 finished with valu

Time: 0:01:51.549570


Model 1a:

In [6]:
def objective_1a(trial, features, target):
    params = {
        'max_depth': trial.suggest_int('max_depth', 2, 32, log=True),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 200),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'bootstrap': True,
        'random_state': 20240627,  
    }

    model = RandomForestClassifier(**params)

    number_folds = 5
    n_splits = 5
    seed = 20240627
    cv = StratifiedShuffleSplit(n_splits=n_splits,
                                test_size=1/n_splits,
                                random_state=seed)
    auc_scores = []
    # cross validation loop for oversample
    for train_idx, val_idx in cv.split(features, target):
        X_train, X_val = features.iloc[train_idx], features.iloc[val_idx]
        y_train, y_val = target[train_idx], target[val_idx]

        # oversample
        oversampler = RandomOverSampler()
        X_train_resampled, y_train_resampled = oversampler.fit_resample(X_train, y_train)

        # use oversample data to train model
        model.fit(X_train_resampled, y_train_resampled)
        
        # calculate auc
        y_proba = model.predict_proba(X_val)[:, 1]
        fold_auc = roc_auc_score(y_val, y_proba)
        auc_scores.append(fold_auc)
    mean_auc = np.array(auc_scores).mean()
    std_err = np.std(auc_scores) / np.sqrt(number_folds)

    trial.set_user_attr("std_err", float(std_err)) 
    return mean_auc

In [7]:
if __name__ == "__main__":
    start_time = datetime.now()
    
    # choose model data
    model_id = "model_1a"
    model_category = "rf"
    n_trials = 50
    
    # load training set
    X_data = pd.read_csv(f"data/X_train_{model_id}.csv", keep_default_na=False)
    y_data = pd.read_csv(f"data/y_train_{model_id}.csv", keep_default_na=False).values.ravel()

    # load study
    study = optuna.create_study(direction="maximize", 
                                sampler=optuna.samplers.TPESampler(seed=2020))
    
    # optimize calculate
    study.optimize(lambda trial: objective_1a(trial, X_data, y_data), n_trials=n_trials, n_jobs=-1)
    
    # store into json file
    all_trials_path = f"results/{model_id}_{model_category}_trials.json"
    with open(all_trials_path, "w", encoding="utf-8") as outfile:
        for trial in study.trials:
            serialized_trial = {
                "number": trial.number,
                "value": trial.value,
                "params": trial.params,
                "user_attrs": trial.user_attrs,
            }
            json.dump(serialized_trial, outfile)
            outfile.write("\n")
    # find the best and store
    best_trial_path = f"results/{model_id}_{model_category}_trial_best.json"
    best_trial = study.best_trial
    serialized_best = {
        "number": best_trial.number,
        "value": best_trial.value,
        "params": best_trial.params,
        "user_attrs": best_trial.user_attrs,
    }
    with open(best_trial_path, "w", encoding="utf-8") as outfile:
        json.dump(serialized_best, outfile)
    print(f"Time: {datetime.now() - start_time}")

[I 2026-05-11 11:01:10,956] A new study created in memory with name: no-name-e823bf28-c1df-466f-8a48-1d1d9fc304cd
[I 2026-05-11 11:01:43,889] Trial 17 finished with value: 0.8159987515605494 and parameters: {'max_depth': 2, 'min_samples_leaf': 18, 'max_features': 'sqrt'}. Best is trial 17 with value: 0.8159987515605494.
[I 2026-05-11 11:01:44,741] Trial 18 finished with value: 0.815435393258427 and parameters: {'max_depth': 2, 'min_samples_leaf': 145, 'max_features': 'sqrt'}. Best is trial 17 with value: 0.8159987515605494.
[I 2026-05-11 11:01:45,284] Trial 15 finished with value: 0.8269805206726886 and parameters: {'max_depth': 5, 'min_samples_leaf': 166, 'max_features': 'sqrt'}. Best is trial 15 with value: 0.8269805206726886.
[I 2026-05-11 11:01:45,670] Trial 0 finished with value: 0.8297904274069177 and parameters: {'max_depth': 5, 'min_samples_leaf': 22, 'max_features': 'log2'}. Best is trial 0 with value: 0.8297904274069177.
[I 2026-05-11 11:01:45,773] Trial 3 finished with value

Time: 0:01:52.282866


Model 2a:

In [8]:
def objective_2a(trial, features, target):
    params = {
        'max_depth': trial.suggest_int('max_depth', 2, 32, log=True),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 200),
        'bootstrap': True,
        'random_state': 20240627,  
    }

    model = RandomForestClassifier(**params)

    number_folds = 5
    n_splits = 5
    seed = 20240627
    cv = StratifiedShuffleSplit(n_splits=n_splits,
                                test_size=1/n_splits,
                                random_state=seed)
    auc_scores = []
    # cross validation loop for oversample
    for train_idx, val_idx in cv.split(features, target):
        X_train, X_val = features.iloc[train_idx], features.iloc[val_idx]
        y_train, y_val = target[train_idx], target[val_idx]

        # oversample
        oversampler = RandomOverSampler()
        X_train_resampled, y_train_resampled = oversampler.fit_resample(X_train, y_train)

        # use oversample data to train model
        model.fit(X_train_resampled, y_train_resampled)
        
        # calculate auc
        y_proba = model.predict_proba(X_val)[:, 1]
        fold_auc = roc_auc_score(y_val, y_proba)
        auc_scores.append(fold_auc)
    mean_auc = np.array(auc_scores).mean()
    std_err = np.std(auc_scores) / np.sqrt(number_folds)

    trial.set_user_attr("std_err", float(std_err)) 
    return mean_auc

In [9]:
if __name__ == "__main__":
    start_time = datetime.now()
    
    # choose model data
    model_id = "model_2a"
    model_category = "rf"
    n_trials = 50
    
    # load training set
    X_data = pd.read_csv(f"data/X_train_{model_id}.csv", keep_default_na=False)
    y_data = pd.read_csv(f"data/y_train_{model_id}.csv", keep_default_na=False).values.ravel()

    # load study
    study = optuna.create_study(direction="maximize", 
                                sampler=optuna.samplers.TPESampler(seed=2020))
    
    # optimize calculate
    study.optimize(lambda trial: objective_2a(trial, X_data, y_data), n_trials=n_trials, n_jobs=-1)
    
    # store into json file
    all_trials_path = f"results/{model_id}_{model_category}_trials.json"
    with open(all_trials_path, "w", encoding="utf-8") as outfile:
        for trial in study.trials:
            serialized_trial = {
                "number": trial.number,
                "value": trial.value,
                "params": trial.params,
                "user_attrs": trial.user_attrs,
            }
            json.dump(serialized_trial, outfile)
            outfile.write("\n")
    # find the best and store
    best_trial_path = f"results/{model_id}_{model_category}_trial_best.json"
    best_trial = study.best_trial
    serialized_best = {
        "number": best_trial.number,
        "value": best_trial.value,
        "params": best_trial.params,
        "user_attrs": best_trial.user_attrs,
    }
    with open(best_trial_path, "w", encoding="utf-8") as outfile:
        json.dump(serialized_best, outfile)
    print(f"Time: {datetime.now() - start_time}")

[I 2026-05-11 11:03:03,274] A new study created in memory with name: no-name-d4c91d37-884a-472c-8977-496a33f14689
[I 2026-05-11 11:03:31,942] Trial 2 finished with value: 0.7386928319986537 and parameters: {'max_depth': 2, 'max_features': 'log2', 'min_samples_leaf': 57}. Best is trial 2 with value: 0.7386928319986537.
[I 2026-05-11 11:03:32,873] Trial 3 finished with value: 0.7557578222515182 and parameters: {'max_depth': 13, 'max_features': 'log2', 'min_samples_leaf': 160}. Best is trial 3 with value: 0.7557578222515182.
[I 2026-05-11 11:03:33,678] Trial 14 finished with value: 0.7551564450303634 and parameters: {'max_depth': 18, 'max_features': 'sqrt', 'min_samples_leaf': 196}. Best is trial 3 with value: 0.7557578222515182.
[I 2026-05-11 11:03:33,827] Trial 17 finished with value: 0.7576400712452492 and parameters: {'max_depth': 8, 'max_features': 'sqrt', 'min_samples_leaf': 145}. Best is trial 17 with value: 0.7576400712452492.
[I 2026-05-11 11:03:33,896] Trial 15 finished with val

Time: 0:01:50.391084


Model 1b:

In [10]:
def objective_1b(trial, features, target):
    params = {
        'max_depth': trial.suggest_int('max_depth', 2, 32, log=True),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 200),
        'bootstrap': True,
        'random_state': 20240627,  
    }

    model = RandomForestClassifier(**params)

    number_folds = 5
    n_splits = 5
    seed = 20240627
    cv = StratifiedShuffleSplit(n_splits=n_splits,
                                test_size=1/n_splits,
                                random_state=seed)
    auc_scores = []
    # cross validation loop for oversample
    for train_idx, val_idx in cv.split(features, target):
        X_train, X_val = features.iloc[train_idx], features.iloc[val_idx]
        y_train, y_val = target[train_idx], target[val_idx]

        # oversample
        oversampler = RandomOverSampler()
        X_train_resampled, y_train_resampled = oversampler.fit_resample(X_train, y_train)

        # use oversample data to train model
        model.fit(X_train_resampled, y_train_resampled)
        
        # calculate auc
        y_proba = model.predict_proba(X_val)[:, 1]
        fold_auc = roc_auc_score(y_val, y_proba)
        auc_scores.append(fold_auc)
    mean_auc = np.array(auc_scores).mean()
    std_err = np.std(auc_scores) / np.sqrt(number_folds)

    trial.set_user_attr("std_err", float(std_err)) 
    return mean_auc

In [11]:
if __name__ == "__main__":
    start_time = datetime.now()
    
    # choose model data
    model_id = "model_1b"
    model_category = "rf"
    n_trials = 50
    
    # load training set
    X_data = pd.read_csv(f"data/X_train_{model_id}.csv", keep_default_na=False)
    y_data = pd.read_csv(f"data/y_train_{model_id}.csv", keep_default_na=False).values.ravel()

    # load study
    study = optuna.create_study(direction="maximize", 
                                sampler=optuna.samplers.TPESampler(seed=2020))
    
    # optimize calculate
    study.optimize(lambda trial: objective_1b(trial, X_data, y_data), n_trials=n_trials, n_jobs=-1)
    
    # store into json file
    all_trials_path = f"results/{model_id}_{model_category}_trials.json"
    with open(all_trials_path, "w", encoding="utf-8") as outfile:
        for trial in study.trials:
            serialized_trial = {
                "number": trial.number,
                "value": trial.value,
                "params": trial.params,
                "user_attrs": trial.user_attrs,
            }
            json.dump(serialized_trial, outfile)
            outfile.write("\n")
    # find the best and store
    best_trial_path = f"results/{model_id}_{model_category}_trial_best.json"
    best_trial = study.best_trial
    serialized_best = {
        "number": best_trial.number,
        "value": best_trial.value,
        "params": best_trial.params,
        "user_attrs": best_trial.user_attrs,
    }
    with open(best_trial_path, "w", encoding="utf-8") as outfile:
        json.dump(serialized_best, outfile)
    print(f"Time: {datetime.now() - start_time}")

[I 2026-05-11 11:04:53,753] A new study created in memory with name: no-name-70302cda-0716-4529-bb31-63f2eaa724ae
[I 2026-05-11 11:05:23,376] Trial 2 finished with value: 0.832692592136441 and parameters: {'max_depth': 2, 'max_features': 'log2', 'min_samples_leaf': 120}. Best is trial 2 with value: 0.832692592136441.
[I 2026-05-11 11:05:23,686] Trial 15 finished with value: 0.8320668875705891 and parameters: {'max_depth': 2, 'max_features': 'log2', 'min_samples_leaf': 150}. Best is trial 2 with value: 0.832692592136441.
[I 2026-05-11 11:05:25,169] Trial 17 finished with value: 0.8453853846303435 and parameters: {'max_depth': 3, 'max_features': 'sqrt', 'min_samples_leaf': 16}. Best is trial 17 with value: 0.8453853846303435.
[I 2026-05-11 11:05:27,135] Trial 14 finished with value: 0.8506403778034576 and parameters: {'max_depth': 4, 'max_features': 'sqrt', 'min_samples_leaf': 108}. Best is trial 14 with value: 0.8506403778034576.
[I 2026-05-11 11:05:28,675] Trial 10 finished with value:

Time: 0:02:21.156563


Model 2b:

In [12]:
def objective_2b(trial, features, target):
    params = {
        'max_depth': trial.suggest_int('max_depth', 2, 32, log=True),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 200),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'bootstrap': True,
        'random_state': 20240627,  
    }

    model = RandomForestClassifier(**params)

    number_folds = 5
    n_splits = 5
    seed = 20240627
    cv = StratifiedShuffleSplit(n_splits=n_splits,
                                test_size=1/n_splits,
                                random_state=seed)
    auc_scores = []
    # cross validation loop for oversample
    for train_idx, val_idx in cv.split(features, target):
        X_train, X_val = features.iloc[train_idx], features.iloc[val_idx]
        y_train, y_val = target[train_idx], target[val_idx]

        # oversample
        oversampler = RandomOverSampler()
        X_train_resampled, y_train_resampled = oversampler.fit_resample(X_train, y_train)

        # use oversample data to train model
        model.fit(X_train_resampled, y_train_resampled)
        
        # calculate auc
        y_proba = model.predict_proba(X_val)[:, 1]
        fold_auc = roc_auc_score(y_val, y_proba)
        auc_scores.append(fold_auc)
    mean_auc = np.array(auc_scores).mean()
    std_err = np.std(auc_scores) / np.sqrt(number_folds)

    trial.set_user_attr("std_err", float(std_err)) 
    return mean_auc

In [13]:
if __name__ == "__main__":
    start_time = datetime.now()
    
    # choose model data
    model_id = "model_2b"
    model_category = "rf"
    n_trials = 50
    
    # load training set
    X_data = pd.read_csv(f"data/X_train_{model_id}.csv", keep_default_na=False)
    y_data = pd.read_csv(f"data/y_train_{model_id}.csv", keep_default_na=False).values.ravel()

    # load study
    study = optuna.create_study(direction="maximize", 
                                sampler=optuna.samplers.TPESampler(seed=2020))
    
    # optimize calculate
    study.optimize(lambda trial: objective_2b(trial, X_data, y_data), n_trials=n_trials, n_jobs=-1)
    
    # store into json file
    all_trials_path = f"results/{model_id}_{model_category}_trials.json"
    with open(all_trials_path, "w", encoding="utf-8") as outfile:
        for trial in study.trials:
            serialized_trial = {
                "number": trial.number,
                "value": trial.value,
                "params": trial.params,
                "user_attrs": trial.user_attrs,
            }
            json.dump(serialized_trial, outfile)
            outfile.write("\n")
    # find the best and store
    best_trial_path = f"results/{model_id}_{model_category}_trial_best.json"
    best_trial = study.best_trial
    serialized_best = {
        "number": best_trial.number,
        "value": best_trial.value,
        "params": best_trial.params,
        "user_attrs": best_trial.user_attrs,
    }
    with open(best_trial_path, "w", encoding="utf-8") as outfile:
        json.dump(serialized_best, outfile)
    print(f"Time: {datetime.now() - start_time}")

[I 2026-05-11 11:07:14,973] A new study created in memory with name: no-name-1612157c-af5b-4895-bdec-a14149f29e0d
[I 2026-05-11 11:07:48,423] Trial 1 finished with value: 0.7859123661339907 and parameters: {'max_depth': 2, 'min_samples_leaf': 36, 'max_features': 'log2'}. Best is trial 1 with value: 0.7859123661339907.
[I 2026-05-11 11:07:49,178] Trial 3 finished with value: 0.7894327580089819 and parameters: {'max_depth': 2, 'min_samples_leaf': 15, 'max_features': 'sqrt'}. Best is trial 3 with value: 0.7894327580089819.
[I 2026-05-11 11:07:49,553] Trial 18 finished with value: 0.7851573486671459 and parameters: {'max_depth': 2, 'min_samples_leaf': 137, 'max_features': 'log2'}. Best is trial 3 with value: 0.7894327580089819.
[I 2026-05-11 11:07:49,674] Trial 0 finished with value: 0.7884620168890082 and parameters: {'max_depth': 2, 'min_samples_leaf': 31, 'max_features': 'sqrt'}. Best is trial 3 with value: 0.7894327580089819.
[I 2026-05-11 11:07:50,765] Trial 19 finished with value: 0.

Time: 0:02:53.144387
